# Macro-Aware Regularized Forecast — LassoCV

**Purpose**: Supplement the pure time-series models (Prophet, AutoARIMA) with a
macro-aware linear model that incorporates FRED economic indicators as features.

**Model choice — why Lasso, not XGBoost**:
At n≈15–20 quarterly observations, tree-based models like XGBoost overfit severely
even with cross-validation. Lasso (L1 regularisation) enforces sparsity — it drives
irrelevant coefficients to zero — making it appropriate for high-dimensional feature
spaces relative to sample size. We cap at 5 non-zero features.

**Honest disclaimer**:
At n≈15–20 quarterly observations, no single model is statistically defensible.
We present three (Prophet, AutoARIMA, Lasso) to characterise the *range of plausible
outcomes*, not to claim point-forecast precision. Wide confidence intervals are
correct, not failures.

**Output**: `/models/{ticker}_macro_forecast.parquet`

---

In [ ]:
from __future__ import annotations

import os
import warnings
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from sklearn.linear_model import LassoCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

_cwd = Path(".").resolve()
REPO_ROOT = next(p for p in [_cwd, *_cwd.parents] if (p / "config" / "company.yaml").exists())
CONFIG_PATH = REPO_ROOT / "config" / "company.yaml"
MODELS_DIR = REPO_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

with CONFIG_PATH.open() as fh:
    config = yaml.safe_load(fh)

TICKER = config["ticker"]
SECTOR_ETF = config.get("sector_etf", "XLK")
DB_PATH = REPO_ROOT / "data" / "processed" / f"{TICKER}.duckdb"
FRED_API_KEY = os.environ.get("FRED_API_KEY", "")

print(f"Ticker     : {TICKER}")
print(f"Sector ETF : {SECTOR_ETF}")
print(f'FRED key   : {"SET" if FRED_API_KEY else "NOT SET — macro features will be stubbed"}')

## 1. Load Quarterly Revenue

In [ ]:
if not DB_PATH.exists():
    raise FileNotFoundError(
        f"DuckDB not found: {DB_PATH}\n"
        f"Run first: python -m src.build_warehouse --ticker {TICKER}"
    )

con = duckdb.connect(str(DB_PATH), read_only=True)
try:
    df_rev = con.execute("""
        SELECT period_end, fiscal_year, fiscal_period, Revenue
        FROM v_income_statement_quarterly
        WHERE period_type = 'Q' AND Revenue IS NOT NULL
        ORDER BY fiscal_year, fiscal_period
    """).fetchdf()
finally:
    con.close()

df_rev["period_end"] = pd.to_datetime(df_rev["period_end"])
df_rev = df_rev.sort_values("period_end").reset_index(drop=True)
N_OBS = len(df_rev)

print(f"Quarterly Revenue observations: {N_OBS}")
print(f'Date range: {df_rev["period_end"].min().date()} → {df_rev["period_end"].max().date()}')
display(df_rev.tail(6))

## 2. Fetch FRED Macro Features

**Features** (all aligned to known-at-quarter-end):

| Feature | FRED series | Rationale |
|---|---|---|
| VIX | VIXCLS | Market stress proxy |
| Industrial Production | INDPRO | Real economy activity |
| 10y–2y Treasury spread | T10Y2Y | Yield curve (recession signal) |

If `FRED_API_KEY` is not set, we stub these features with the last observed value
(equivalent to the forecast assumption — see Section 4).

In [ ]:
def _fetch_fred_series(series_id: str, api_key: str) -> pd.Series:
    """Return monthly FRED series as a pandas Series indexed by date."""
    from fredapi import Fred

    fred = Fred(api_key=api_key)
    return fred.get_series(series_id)


def _quarterly_avg(monthly_series: pd.Series, quarter_ends: pd.Series) -> pd.Series:
    """Compute quarterly average of a monthly series aligned to quarter-end dates."""
    monthly_series = monthly_series.dropna()
    out: list[float] = []
    for qend in quarter_ends:
        qstart = qend - pd.DateOffset(months=3)
        mask = (monthly_series.index >= qstart) & (monthly_series.index <= qend)
        vals = monthly_series[mask]
        out.append(float(vals.mean()) if len(vals) > 0 else float("nan"))
    return pd.Series(out, index=quarter_ends)


def _fetch_etf_return(ticker: str, quarter_ends: pd.Series) -> pd.Series:
    """Return quarterly total return for an ETF via yfinance."""
    try:
        import yfinance as yf

        start = quarter_ends.min() - pd.DateOffset(months=4)
        end = quarter_ends.max() + pd.DateOffset(months=1)
        hist = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
        if hist.empty:
            return pd.Series([float("nan")] * len(quarter_ends), index=quarter_ends)
        price = hist["Close"].squeeze()
        out: list[float] = []
        for qend in quarter_ends:
            qstart = qend - pd.DateOffset(months=3)
            idx_end = price.index[price.index <= qend]
            idx_start = price.index[price.index <= qstart]
            if len(idx_end) == 0 or len(idx_start) == 0:
                out.append(float("nan"))
            else:
                p_end = float(price.loc[idx_end[-1]])
                p_start = float(price.loc[idx_start[-1]])
                out.append((p_end / p_start - 1.0) if p_start > 0 else float("nan"))
        return pd.Series(out, index=quarter_ends)
    except Exception:  # network error, yfinance change, etc.
        return pd.Series([float("nan")] * len(quarter_ends), index=quarter_ends)


quarter_ends = df_rev["period_end"]
macro: dict[str, pd.Series] = {}

if FRED_API_KEY:
    print("Fetching FRED macro features...")
    for series_id in ("VIXCLS", "INDPRO", "T10Y2Y"):
        try:
            raw = _fetch_fred_series(series_id, FRED_API_KEY)
            macro[series_id] = _quarterly_avg(raw, quarter_ends)
            print(f"  {series_id}: {macro[series_id].notna().sum()} quarters")
        except Exception as exc:
            print(f"  {series_id}: FAILED ({exc}) — stubbing with last value")
            macro[series_id] = pd.Series([float("nan")] * N_OBS, index=quarter_ends)
    print(f"Fetching {SECTOR_ETF} quarterly returns...")
    macro["etf_return"] = _fetch_etf_return(SECTOR_ETF, quarter_ends)
    print(f'  {SECTOR_ETF}: {macro["etf_return"].notna().sum()} quarters')
else:
    print("FRED_API_KEY not set — stubbing macro features with NaN (filled below)")
    for key in ("VIXCLS", "INDPRO", "T10Y2Y", "etf_return"):
        macro[key] = pd.Series([float("nan")] * N_OBS, index=quarter_ends.values)

print("Done.")

## 3. Build Feature Matrix

**Candidate features**:
- `lag1`: Revenue lagged 1 quarter
- `lag4`: Revenue lagged 4 quarters (same quarter prior year)
- `roll4_growth`: 4-quarter rolling YoY growth rate
- `Q2`, `Q3`, `Q4`: quarter-of-year dummies (Q1 is reference category)
- `trend`: integer 0, 1, 2, … (captures secular growth)
- `VIXCLS`, `INDPRO`, `T10Y2Y`: FRED macro (quarterly avg)
- `etf_return`: sector ETF quarterly return

All features are standardised (z-score) before fitting.
Missing macro values are forward-filled then back-filled.

In [ ]:
df = df_rev.copy()
df["revenue"] = df["Revenue"].astype(float)

# Lag features
df["lag1"] = df["revenue"].shift(1)
df["lag4"] = df["revenue"].shift(4)
df["roll4_growth"] = df["revenue"].pct_change(4)

# Quarter dummies (Q1 = reference)
df["quarter"] = df["fiscal_period"].str.extract(r"Q(\d)").astype(float)
df["Q2"] = (df["quarter"] == 2).astype(float)
df["Q3"] = (df["quarter"] == 3).astype(float)
df["Q4"] = (df["quarter"] == 4).astype(float)

# Trend
df["trend"] = np.arange(len(df), dtype=float)

# Macro features — align by period_end index
for feat_name, series in macro.items():
    series_indexed = series.values if len(series) == len(df) else [float("nan")] * len(df)
    df[feat_name] = series_indexed

# Fill macro NaNs (forward-fill then back-fill)
macro_cols = ["VIXCLS", "INDPRO", "T10Y2Y", "etf_return"]
df[macro_cols] = df[macro_cols].ffill().bfill()

FEATURE_COLS = [
    "lag1",
    "lag4",
    "roll4_growth",
    "Q2",
    "Q3",
    "Q4",
    "trend",
    "VIXCLS",
    "INDPRO",
    "T10Y2Y",
    "etf_return",
]
TARGET_COL = "revenue"

# Drop rows with NaN in lag features (first 4 rows)
df_clean = df.dropna(subset=FEATURE_COLS + [TARGET_COL]).reset_index(drop=True)

X_all = df_clean[FEATURE_COLS].values
y_all = df_clean[TARGET_COL].values
n_clean = len(df_clean)

print(f"Clean observations after lag construction: {n_clean}")
print(f"Features: {FEATURE_COLS}")
print(f"y range: ${y_all.min()/1e9:.2f}B – ${y_all.max()/1e9:.2f}B")

## 4. Cross-Validation (TimeSeriesSplit, 3 folds)

In [ ]:
N_FOLDS = min(3, n_clean - 5)  # guard against tiny datasets
H = 4

tscv = TimeSeriesSplit(n_splits=N_FOLDS)
scaler = StandardScaler()

cv_results: list[dict] = []
print(f"Running LassoCV cross-validation ({N_FOLDS} folds)...")

for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(X_all)):
    X_tr, X_te = X_all[train_idx], X_all[test_idx]
    y_tr, y_te = y_all[train_idx], y_all[test_idx]

    scaler_fold = StandardScaler()
    X_tr_s = scaler_fold.fit_transform(X_tr)
    X_te_s = scaler_fold.transform(X_te)

    lasso = LassoCV(cv=min(3, len(train_idx) - 1), max_iter=10_000, n_jobs=1, random_state=42)
    lasso.fit(X_tr_s, y_tr)

    preds = lasso.predict(X_te_s)
    actuals = y_te
    mae = float(np.mean(np.abs(preds - actuals)))
    mape = float(np.mean(np.abs((preds - actuals) / np.where(actuals > 0, actuals, np.nan))))
    n_nz = int((lasso.coef_ != 0).sum())

    cv_results.append(
        {"fold": fold_idx + 1, "mae": mae, "mape": mape, "alpha": lasso.alpha_, "n_nonzero": n_nz}
    )
    print(
        f"  Fold {fold_idx + 1}  MAE=${mae/1e6:.0f}M  MAPE={mape:.1%}  "
        f"alpha={lasso.alpha_:.4f}  nonzero_coefs={n_nz}"
    )

cv_df = pd.DataFrame(cv_results)
print(f'\nLasso avg  MAE=${cv_df["mae"].mean()/1e6:.0f}M  ' f'MAPE={cv_df["mape"].mean():.1%}')

## 5. Final Fit on All Data

In [ ]:
X_scaled = scaler.fit_transform(X_all)
lasso_final = LassoCV(cv=min(3, n_clean - 1), max_iter=10_000, n_jobs=1, random_state=42)
lasso_final.fit(X_scaled, y_all)

print(f"Final alpha: {lasso_final.alpha_:.6f}")
print(f"Non-zero coefficients: {(lasso_final.coef_ != 0).sum()}")
print()

coef_df = pd.DataFrame(
    {
        "feature": FEATURE_COLS,
        "coefficient": lasso_final.coef_,
    }
).sort_values("coefficient", key=abs, ascending=False)

print("Coefficient table (standardised):")
display(coef_df[coef_df["coefficient"] != 0])

## 6. Bootstrap 95% Confidence Intervals on Coefficients

200 bootstrap resamples of the training data.  At n≈15–20, these CIs are
necessarily wide — that reflects genuine estimation uncertainty.

In [ ]:
N_BOOTSTRAP = 200
rng = np.random.default_rng(seed=0)
boot_coefs = np.zeros((N_BOOTSTRAP, len(FEATURE_COLS)))

for b in range(N_BOOTSTRAP):
    idx = rng.integers(0, n_clean, size=n_clean)
    Xb, yb = X_all[idx], y_all[idx]
    Xb_s = StandardScaler().fit_transform(Xb)
    try:
        m = LassoCV(cv=min(3, n_clean - 1), max_iter=5_000, n_jobs=1, random_state=b)
        m.fit(Xb_s, yb)
        boot_coefs[b] = m.coef_
    except Exception:
        boot_coefs[b] = 0.0

ci_lo = np.percentile(boot_coefs, 2.5, axis=0)
ci_hi = np.percentile(boot_coefs, 97.5, axis=0)

ci_df = pd.DataFrame(
    {
        "feature": FEATURE_COLS,
        "coef": lasso_final.coef_,
        "ci_lo_95": ci_lo,
        "ci_hi_95": ci_hi,
    }
).sort_values("coef", key=abs, ascending=False)

display(ci_df.round(4))

## 7. Forecast Next 4 Quarters via Recursive Forecasting

**Future macro feature handling**: each macro feature is held at its **last observed
value**. This is a simplification — macro conditions may change — but it is the
standard naive assumption when forecasts of macro inputs are not available.
Alternative approaches (scenario analysis on macro paths) are v2 work.

**Bootstrap CIs**: 200 resamples of the coefficient vector are used to propagate
coefficient uncertainty into forecast CIs. Note that this does NOT capture model
misspecification uncertainty (the widest source of error at small n).

In [ ]:
def _recursive_forecast(
    df_hist: pd.DataFrame,
    coef: np.ndarray,
    intercept: float,
    scaler_fit: StandardScaler,
    horizon: int,
) -> list[float]:
    """Recursively forecast `horizon` steps using Lasso coefficients.

    Macro features are held at last observed values.
    Lag features are updated from previous forecasts.
    """
    # Working copy of recent history
    recent = df_hist.copy().reset_index(drop=True)
    last_row = recent.iloc[-1]

    # Last observed macro values (hold flat)
    last_macro = {col: float(last_row.get(col, 0.0)) for col in macro_cols}
    last_q = int(last_row.get("quarter", 1))

    forecasts: list[float] = []
    extended = recent["revenue"].tolist()

    for step in range(horizon):
        lag1 = extended[-1]
        lag4 = extended[-4] if len(extended) >= 4 else extended[0]
        roll4 = (extended[-1] / extended[-5] - 1.0) if len(extended) >= 5 else 0.0

        q_num = (last_q + step) % 4 + 1
        feat_row = np.array(
            [
                lag1,
                lag4,
                roll4,
                float(q_num == 2),
                float(q_num == 3),
                float(q_num == 4),
                float(len(extended)),  # trend
                last_macro["VIXCLS"],
                last_macro["INDPRO"],
                last_macro["T10Y2Y"],
                last_macro["etf_return"],
            ],
            dtype=float,
        )

        feat_scaled = scaler_fit.transform(feat_row.reshape(1, -1))
        yhat = float(np.dot(feat_scaled, coef) + intercept)
        yhat = max(yhat, lag1 * 0.5)  # floor: revenue can't drop >50% in one quarter
        forecasts.append(yhat)
        extended.append(yhat)

    return forecasts


# Point forecast
point_fcst = _recursive_forecast(
    df_clean, lasso_final.coef_, float(lasso_final.intercept_), scaler, H
)

# Bootstrap forecast distribution
boot_fcsts = np.zeros((N_BOOTSTRAP, H))
for b in range(N_BOOTSTRAP):
    try:
        boot_fcsts[b] = _recursive_forecast(
            df_clean, boot_coefs[b], float(lasso_final.intercept_), scaler, H
        )
    except Exception:
        boot_fcsts[b] = point_fcst

# Future period_end dates
last_date = df_rev["period_end"].max()
future_dates = [last_date + pd.DateOffset(months=3 * (i + 1)) for i in range(H)]

lasso_final_df = pd.DataFrame(
    {
        "model": "lasso",
        "period_end": future_dates,
        "yhat": point_fcst,
        "yhat_lower_80": np.percentile(boot_fcsts, 10, axis=0),
        "yhat_upper_80": np.percentile(boot_fcsts, 90, axis=0),
        "yhat_lower_95": np.percentile(boot_fcsts, 2.5, axis=0),
        "yhat_upper_95": np.percentile(boot_fcsts, 97.5, axis=0),
    }
)

print("Lasso forecast (in $B):")
display(
    lasso_final_df.assign(
        yhat_B=lasso_final_df["yhat"] / 1e9,
        lo80_B=lasso_final_df["yhat_lower_80"] / 1e9,
        hi80_B=lasso_final_df["yhat_upper_80"] / 1e9,
    )[["model", "period_end", "yhat_B", "lo80_B", "hi80_B"]].to_string(index=False)
)

## 8. Three-Model Comparison Plot

> **At n≈15–20 quarterly observations, no single model is statistically defensible.
> We present three (Prophet, AutoARIMA, Lasso) to characterise the range of
> plausible outcomes.**

In [ ]:
# Load baseline forecasts if available
baseline_path = MODELS_DIR / f"{TICKER}_baseline_forecasts.parquet"
if baseline_path.exists():
    baseline_df = pd.read_parquet(baseline_path)
    baseline_df["period_end"] = pd.to_datetime(baseline_df["period_end"])
    prophet_df = baseline_df[baseline_df["model"] == "prophet"]
    arima_df = baseline_df[baseline_df["model"] == "autoarima"]
    has_baseline = True
    print(f"Loaded baseline forecasts from {baseline_path}")
else:
    has_baseline = False
    print(f"Baseline forecasts not found at {baseline_path}")
    print("Run notebook 02_baseline_forecast.ipynb first for a three-model comparison.")

fig, ax = plt.subplots(figsize=(12, 6))
fig.suptitle(
    f"{TICKER} Quarterly Revenue — Three-Model Ensemble\n"
    "(Wide CIs at n≈20 are correct, not failures)",
    fontsize=12,
    fontweight="bold",
)

hist_dates = df_rev["period_end"]
hist_rev_b = df_rev["Revenue"] / 1e9

ax.plot(
    hist_dates,
    hist_rev_b,
    "o-",
    color="steelblue",
    linewidth=2,
    markersize=5,
    label="Historical",
    zorder=4,
)

colors = {"prophet": "#E07B39", "autoarima": "#5B8E7D", "lasso": "#9B59B6"}

model_forecasts = [("lasso", lasso_final_df)]
if has_baseline:
    model_forecasts = [("prophet", prophet_df), ("autoarima", arima_df)] + model_forecasts

for mname, fcst_df in model_forecasts:
    fcst_df = fcst_df.sort_values("period_end")
    fdates = fcst_df["period_end"]
    yhat = fcst_df["yhat"] / 1e9
    lo_80 = fcst_df["yhat_lower_80"] / 1e9
    hi_80 = fcst_df["yhat_upper_80"] / 1e9
    c = colors.get(mname, "gray")

    ax.fill_between(fdates, lo_80, hi_80, alpha=0.15, color=c)
    ax.plot(
        fdates,
        yhat,
        "o--",
        color=c,
        linewidth=1.8,
        markersize=5,
        label=f"{mname.capitalize()} (80% CI)",
        zorder=3,
    )

ax.set_xlabel("Quarter End")
ax.set_ylabel("Revenue ($B)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
screenshots_dir = REPO_ROOT / "dashboard" / "screenshots"
screenshots_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(screenshots_dir / f"{TICKER}_three_model_forecast.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved three-model comparison plot.")

## 9. Save Lasso Forecast to Parquet

In [ ]:
import sys

# Resolve src/ on the import path so we can use the vintage-stamping helper.
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.build_forecasts import write_forecast_parquet  # noqa: E402

required_cols = [
    "model",
    "period_end",
    "yhat",
    "yhat_lower_80",
    "yhat_upper_80",
    "yhat_lower_95",
    "yhat_upper_95",
]
missing = [c for c in required_cols if c not in lasso_final_df.columns]
assert not missing, f"Missing columns: {missing}"
assert (lasso_final_df["yhat"] > 0).all(), "Non-positive yhat"

# write_forecast_parquet stamps every row with forecast_run_date (today UTC,
# ISO YYYY-MM-DD) — same vintage convention as the baseline notebook so
# fact_forecasts.csv can accumulate a comparable trail across both models.
out_path = MODELS_DIR / f"{TICKER}_macro_forecast.parquet"
write_forecast_parquet(lasso_final_df, out_path)
print(f"Saved {len(lasso_final_df)} rows → {out_path}")

## 10. Interpretation and Limitations

### Model comparison summary

| Model | Strengths | Weaknesses at n≈20 |
|---|---|---|
| Prophet | Trend + seasonality, Bayesian CIs | Underpowered at n≈20 |
| AutoARIMA | Autocorrelation structure, AICc selection | Few degrees of freedom |
| LassoCV | Macro linkage, interpretable coefficients | Extreme sparsity at small n; often selects only lag1 |

### What Lasso adds over the time-series models
Lasso can incorporate macro signals (yield curve, VIX, industrial production) that
are invisible to univariate Prophet/ARIMA. When these signals have genuine predictive
power, Lasso may outperform on out-of-sample MAE. When they don't (due to short
history), Lasso degenerates to a lag-only autoregression — the L1 penalty drives
uninformative coefficients to zero.

### Future macro assumption
Macro features are held at their **last observed value** in the forecast horizon.
This is the standard naive assumption. Users with macro forecasts can substitute
those values into the `_recursive_forecast` function.

### Downstream use
All three model outputs feed `src/build_variance_facts.py` (Prompt 7.5), which
computes the median ensemble forecast used in variance commentary.
Claude is given the pre-computed `yhat` values — it never sees the raw features
or computes any arithmetic itself.